# บทที่ 3: สถาปัตยกรรม Multi-Layer Perceptron (MLP)

ใน Notebook นี้ เราจะเรียนรู้โครงสร้างของ MLP, Forward Propagation, การแก้ปัญหา XOR และการคำนวณ Parameters

## 1. นำเข้าไลบรารีที่จำเป็น

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

from matplotlib.colors import ListedColormap

## 2. ข้อจำกัดของ Perceptron เดี่ยวและปัญหา XOR

In [ ]:
# แสดงปัญหา XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

plt.figure(figsize=(8, 6))
colors = ['red', 'blue']
for i, (xi, yi) in enumerate(zip(X, y_xor)):
    plt.scatter(xi[0], xi[1], c=colors[yi], s=200, edgecolors='black', linewidths=2)
    plt.annotate(f'({xi[0]}, {xi[1]}) → {yi}', 
                 xy=(xi[0], xi[1]), xytext=(10, 10), 
                 textcoords='offset points', fontsize=12)

plt.xlabel('x₁')
plt.ylabel('x₂')
plt.title('ปัญหา XOR: ไม่สามารถแยกด้วยเส้นตรง')
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.show()

print("สังเกต: ไม่มีเส้นตรงที่สามารถแยก class 0 และ class 1 ได้")

## 3. โครงสร้างของ MLP

MLP ประกอบด้วย:
- Input Layer
- Hidden Layer(s)
- Output Layer

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

class MLP:
    """
    Multi-Layer Perceptron Implementation
    """
    def __init__(self, layer_sizes, learning_rate=0.5):
        """
        Parameters:
        - layer_sizes: list ของจำนวน neurons ในแต่ละ layer
          เช่น [2, 4, 1] = 2 inputs, 4 hidden, 1 output
        """
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.n_layers = len(layer_sizes)
        
        # Initialize weights and biases
        self.weights = []
        self.biases = []
        
        for i in range(self.n_layers - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * np.sqrt(2/layer_sizes[i])
            b = np.zeros((layer_sizes[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)
            
    def forward(self, x):
        """Forward propagation"""
        self.activations = [x.reshape(-1, 1)]
        self.z_values = []
        
        current = self.activations[0]
        for i in range(self.n_layers - 1):
            z = np.dot(self.weights[i], current) + self.biases[i]
            self.z_values.append(z)
            current = sigmoid(z)
            self.activations.append(current)
            
        return current
    
    def backward(self, y):
        """Backpropagation"""
        y = y.reshape(-1, 1)
        m = 1
        
        # Output layer error
        delta = self.activations[-1] - y
        
        self.dW = []
        self.db = []
        
        for i in range(self.n_layers - 2, -1, -1):
            dW = np.dot(delta, self.activations[i].T)
            db = delta
            
            self.dW.insert(0, dW)
            self.db.insert(0, db)
            
            if i > 0:
                delta = np.dot(self.weights[i].T, delta) * sigmoid_derivative(self.z_values[i-1])
                
    def update_weights(self):
        """Update weights using gradient descent"""
        for i in range(self.n_layers - 1):
            self.weights[i] -= self.lr * self.dW[i]
            self.biases[i] -= self.lr * self.db[i]
            
    def train(self, X, y, epochs=10000, verbose=True):
        """Train the network"""
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                self.forward(xi)
                self.backward(yi)
                self.update_weights()
                
            if verbose and epoch % 2000 == 0:
                loss = self.compute_loss(X, y)
                print(f"Epoch {epoch}: Loss = {loss:.6f}")
                
    def predict(self, x):
        """Predict"""
        return self.forward(x)[0, 0]
    
    def compute_loss(self, X, y):
        """Compute MSE loss"""
        total_loss = 0
        for xi, yi in zip(X, y):
            pred = self.predict(xi)
            total_loss += (yi - pred)**2
        return total_loss / len(X)

## 4. การแก้ปัญหา XOR ด้วย MLP

In [ ]:
# สร้าง MLP สำหรับ XOR
mlp = MLP(layer_sizes=[2, 4, 1], learning_rate=0.5)

# Train
print("=== Training MLP for XOR ===")
mlp.train(X, y_xor, epochs=10000)

# Test
print("\n=== Results ===")
for xi, yi in zip(X, y_xor):
    pred = mlp.predict(xi)
    print(f"Input: {xi}, Target: {yi}, Prediction: {pred:.4f} → {1 if pred > 0.5 else 0}")

## 5. แสดง Decision Boundary

In [ ]:
def plot_decision_boundary(mlp, X, y, title):
    """Plot decision boundary"""
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    
    Z = np.array([mlp.predict(np.array([x, y])) 
                  for x, y in zip(xx.ravel(), yy.ravel())])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['red', 'blue']))
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    
    for i, (xi, yi) in enumerate(zip(X, y)):
        plt.scatter(xi[0], xi[1], c=['red', 'blue'][yi], s=200, 
                   edgecolors='black', linewidths=2)
        
    plt.xlabel('x₁')
    plt.ylabel('x₂')
    plt.title(title)
    plt.show()

plot_decision_boundary(mlp, X, y_xor, 'Decision Boundary ของ MLP สำหรับ XOR')

## 6. การคำนวณจำนวน Parameters

In [ ]:
def count_parameters(layer_sizes):
    """
    คำนวณจำนวน parameters ใน MLP
    
    Parameters = weights + biases
    - weights: n_in × n_out
    - biases: n_out
    """
    total = 0
    print(f"Layer sizes: {layer_sizes}")
    print("\n" + "="*50)
    
    for i in range(len(layer_sizes) - 1):
        n_in = layer_sizes[i]
        n_out = layer_sizes[i+1]
        
        weights = n_in * n_out
        biases = n_out
        layer_params = weights + biases
        total += layer_params
        
        print(f"Layer {i} → Layer {i+1}:")
        print(f"  Weights: {n_in} × {n_out} = {weights}")
        print(f"  Biases: {biases}")
        print(f"  Subtotal: {layer_params}")
        print("-" * 30)
        
    print(f"\n>>> Total Parameters: {total}")
    return total

# Example
print("=== MLP สำหรับ XOR ===")
_ = count_parameters([2, 4, 1])

print("\n\n=== MLP ขนาดใหญ่ ===")
_ = count_parameters([784, 128, 64, 10])

## 7. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: Forward Propagation
ให้ MLP มีโครงสร้าง [2, 3, 1] โดย:
- W1 = [[0.5, 0.2], [0.3, 0.4], [0.1, 0.6]]
- b1 = [[0.1], [0.2], [0.3]]
- W2 = [[0.4, 0.3, 0.2]]
- b2 = [[0.1]]

จงคำนวณผลลัพธ์สำหรับ input x = [1, 0]

In [ ]:
# เขียนโค้ดที่นี่
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.array([[1], [0]])
W1 = np.array([[0.5, 0.2], [0.3, 0.4], [0.1, 0.6]])
b1 = np.array([[0.1], [0.2], [0.3]])
W2 = np.array([[0.4, 0.3, 0.2]])
b2 = np.array([[0.1]])

# Forward pass
z1 = np.dot(W1, x) + b1
a1 = sigmoid(z1)
z2 = np.dot(W2, a1) + b2
a2 = sigmoid(z2)

print(f"Input: x = [1, 0]")
print(f"\nz1 = W1·x + b1 =\n{z1}")
print(f"\na1 = sigmoid(z1) =\n{a1}")
print(f"\nz2 = W2·a1 + b2 =\n{z2}")
print(f"\nOutput: a2 = sigmoid(z2) = {a2[0,0]:.6f}")

### แบบฝึกหัดที่ 2: คำนวณจำนวน Parameters
จงคำนวณจำนวน parameters ของ MLP ที่มีโครงสร้าง:
- Input: 10 neurons
- Hidden 1: 20 neurons
- Hidden 2: 15 neurons
- Output: 3 neurons

In [ ]:
# เขียนโค้ดที่นี่
layer_sizes = [10, 20, 15, 3]
total = count_parameters(layer_sizes)

### แบบฝึกหัดที่ 3: ฝึก MLP สำหรับ AND Gate

In [ ]:
# เขียนโค้ดที่นี่
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

mlp_and = MLP(layer_sizes=[2, 2, 1], learning_rate=0.5)
mlp_and.train(X, y_and, epochs=5000)

print("\n=== AND Gate Results ===")
for xi, yi in zip(X, y_and):
    pred = mlp_and.predict(xi)
    print(f"Input: {xi}, Target: {yi}, Prediction: {pred:.4f}")

### แบบฝึกหัดที่ 4: ฝึก MLP สำหรับ OR Gate

In [ ]:
# เขียนโค้ดที่นี่
y_or = np.array([0, 1, 1, 1])

mlp_or = MLP(layer_sizes=[2, 2, 1], learning_rate=0.5)
mlp_or.train(X, y_or, epochs=5000)

print("\n=== OR Gate Results ===")
for xi, yi in zip(X, y_or):
    pred = mlp_or.predict(xi)
    print(f"Input: {xi}, Target: {yi}, Prediction: {pred:.4f}")

## บทสรุป

Notebook นี้แสดงให้เห็นว่า:
1. **Perceptron เดี่ยว** ไม่สามารถแก้ปัญหา XOR ได้
2. **MLP** สามารถแก้ปัญหา XOR ได้ เพราะมี hidden layer
3. **การคำนวณ parameters** สำคัญสำหรับการวางแผนโครงสร้างโมเดล

ผู้อ่านสามารถทดลองเปลี่ยนจำนวน hidden neurons และ learning rate